# Inspect ConvUMM-GAN, ConvUMM, ConformerUMM Ablation
@hanoihantrakul 7JUN2024
If you are looking at this notebook after JUN2024 I have reduced the filesize by running cells that plot a lot of images and audio with incorrect variables so that the images/audio will not be saved in file. Just run the notebook in order and these errors will disappear.

@hanoihantrakul 2JUN2024
Perform a quick ablation on the mel reconstruction accuracy to see the effect of ConvUMM-GAN and regular ConvUMM/ConformerUMM on the dataset. 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torchaudio
import IPython.display as ipd
assert torch.cuda.is_available()
import sys
import os
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
os.chdir('/opt/tiger/samantha')

# Load Dataset

In [ ]:
from recipes.datasets.mcc.mix_mkii import MixDataModule

In [ ]:
# see
# samantha/recipes/umm/conf/convumm_gan/convumm_gan_719M_25hz_vocals_2255_2375.yaml
SAMPLE_RATE = 24000
HOP_LENGTH = 240
BATCH_SIZE = 8
SHUFFLE_BUFFER_SIZE = 10
MIN_DURATION = 1
MAX_DURATION = 60
TOKENIZER = None
FRAME_RATE = 25

DATA_IDS = [2255]
DATA_WEIGHTS = [1]

mdm = MixDataModule(
    data_ids=DATA_IDS,
    data_weights=DATA_WEIGHTS,
    batch_size=BATCH_SIZE * MAX_DURATION * SAMPLE_RATE, # means "total number of audio samples loaded into memory"
    shuffle_buffer_size=SHUFFLE_BUFFER_SIZE,
    sample_rate=SAMPLE_RATE,
    min_duration=MIN_DURATION,
    max_duration=MAX_DURATION,
    frame_rate=FRAME_RATE,
    tokenizer=TOKENIZER,
)

# There will be many printouts if loading for the first time

In [ ]:
dm = mdm.train_dataloader()
dm_it = iter(dm)

In [ ]:
NUM_ITERATIONS=1

audio_list = []
token_list = []
for i in range(NUM_ITERATIONS):
    batch = next(dm_it)
    print(batch.keys())
    audio_list.append(batch['audio'])
    token_list.append(batch['token'])

In [ ]:
print(len(audio_list))
print(audio_list[0].shape)
print(len(token_list))
print(token_list[0].shape)

# Define Models for Testing

In [ ]:
from recipes.umm.requires.model_initializer import (init_stage3_conv1d, init_stage3, init_convumm_gan)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model_convumm_gan(ckpt_path, cache_dir):
    print(f"Downloading {ckpt_path}")
    DUMMY_RANK = 0
    token_model = init_convumm_gan(ckpt_path, DUMMY_RANK, cache_dir)[
        "convumm_gan_model"
    ].eval()
    return token_model

def load_model_convumm_conv1D(ckpt_path, cache_dir):
    print(f"Downloading {ckpt_path}")
    DUMMY_RANK = 0
    token_model = init_stage3_conv1d(ckpt_path, DUMMY_RANK, cache_dir)[
        "Stage3Conv1D"
    ].eval()
    return token_model

def load_model_conformer(ckpt_path, cache_dir):
    print(f"Downloading {ckpt_path}")
    DUMMY_RANK = 0
    token_model = init_stage3(ckpt_path, DUMMY_RANK, cache_dir)[
        "Stage3"
    ].eval()
    return token_model

def load_model_convumm_pitch(ckpt_path, cache_dir):
    print(f"Downloading {ckpt_path}")
    DUMMY_RANK = 0
    token_model = init_stage3_conv1d(ckpt_path, DUMMY_RANK, cache_dir)[
        "Stage3Conv1D"
    ].eval()
    return token_model 

MODELS_DICT = {
    # GROUP A
    # @hanoihantrakul: 2JUN2024 This is the latest ConvUMM-GAN trained on 2255 2375
    "convumm_gan_2255_2375": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/convumm_gan_master/convumm_gan_719M_2255_2375_vocals_bert-base-multilingual-uncased_EMAEntropy32768x32/checkpoints/step=0590000.ckpt",
    
    # GROUP B
    # @hanoihantrakul: 22APR2024 This is the latest Direct Stage3 ConvUMM trained on 2255 2375
    "convumm_direct_stage3_2255_2375": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conv_mixedZHEN/umm_stage3_conv1D_v2_2250mixedZHEN_2375Speech_direct_stage3_optim_mem_EMAVQ32768x32/checkpoints/step=0310000.ckpt",
    # @hanoihantrakul: 26APR2024 This is the latest ConformerUMM trained on 2255 2375
    "conformerumm_2255_2375": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conformer_2255mixedZHEN_2375Speech/umm_stage3_2255mixedZHEN_2375Speech_optim_mem_bert-base-multilingual-uncased_None32768x32/checkpoints/step=0180000.ckpt",
    
    # Group C
    # @hanoihantrakul: OCT2023 Legacy ConformerUMM that was reference baseline for QQ L2S ZH
    "legacy_conformerumm_l2s_zh": "hdfs://haruna/home/byte_speech_sv/zongyu.yin/logs/umm/umm_stage3_zh_dw1-1-0_wordpiece_vq32768x16-layer12/checkpoints/step=070000.ckpt",
    # @hanoihantrakul: OCT2023 Legacy ConformerUMM that was reference baseline for QQ L2S ZH
    "legacy_conformerumm_svs": "hdfs:///home/byte_speech_sv/zongyu.yin/logs/umm_mix/umm_stage3_preclipped_bert-base-multilingual-uncased_None32768x32/checkpoints/step=0330000.ckpt",
    
    # Group D
    # @hanoihantrakul: 2JUNE2024 Supervised Pitch Loss (SPL)
    "convumm_spl": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conv_pitch_losses/umm_direct_stage3_conv1d_2255_2375_supervised_pitch_loss_EMAVQ32768x32/checkpoints/step=0500000.ckpt",
    # @hanoihantrakul: 2JUNE2024 Perceptual Pitch Loss (PPL)
    "convumm_ppl": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conv_pitch_losses/umm_direct_stage3_conv1d_2255_2375_perceptual_pitch_loss_EMAVQ32768x32/checkpoints/step=0450000.ckpt",
    # @hanoihantrakul: 2JUNE2024 Supervised + Perceptual Pitch Loss (SPL+PPL)
    "convumm_spl_ppl": "hdfs://haruna/home/byte_data_seed/lf_lq/speech/user/hanoi.hantrakul/logs/umm_conv_pitch_losses/umm_direct_stage3_conv1d_2255_2375_supervised_and_perceptual_pitch_loss_EMAVQ32768x32/checkpoints/step=0470000.ckpt"
}

# Inspect ConvUMM-GAN

In [ ]:
MODEL_KEY = "convumm_gan_2255_2375"
ckpt_path = MODELS_DICT[MODEL_KEY]
unique_cache_dir = "./." + MODEL_KEY

In [ ]:
# Load model
token_model_convumm_gan = load_model_convumm_gan(ckpt_path, unique_cache_dir)

In [ ]:
audio_input = audio_list[0]
print(audio_input.shape)

In [ ]:
# Extract tokens

# this function needs some preprocessing to the audio based on the model definition
tokens = token_model_convumm_gan.wav2token(token_model_convumm_gan._prepare_wav(audio_input).to(device))
print(tokens.shape)

# Convert tokens to mel-160 spectrogram
mel = token_model_convumm_gan.token2mel(tokens)
print(mel.shape)

In [ ]:
# Plot for ConvUMM-GAN
NUM_PLOTS = 5
for i in range(NUM_PLOTS):
    plt.figure(figsize=(16, 8))
    plt.pcolor(mel[i].cpu().T, vmin=-6, vmax=0.5)
    plt.title(MODEL_KEY + " Mel-160 Spec Reconstruction 2255 2375")

## Plot Ground Truth (based on ConvUMM-GAN lit module processing)

In [ ]:
from recipes.umm.utils.mel_utils import torch_wav2spec
gt_mel = torch_wav2spec(token_model_convumm_gan._prepare_wav(audio_input).to(device))
print(gt_mel.shape)

In [ ]:
NUM_PLOTS = 5
for i in range(NUM_PLOTS):
    plt.figure(figsize=(16, 8))
    plt.pcolor(gt_mel[i].cpu().T, vmin=-6, vmax=0.5)
    plt.title("Grount Truth Mel-160 Spec")

In [ ]:
audios_gt_dualumm = token_model_convumm_gan._prepare_wav(audio_input)

In [ ]:
ipd.Audio(audios_gt_dualumm[0], rate=SAMPLE_RATE)

In [ ]:
ipd.Audio(audios_gt_dualumm[1], rate=SAMPLE_RATE)

In [ ]:
ipd.Audio(audios_gt_dualumm[2], rate=SAMPLE_RATE)

In [ ]:
ipd.Audio(audios_gt_dualumm[3], rate=SAMPLE_RATE)

In [ ]:
ipd.Audio(audios_gt_dualumm[4], rate=SAMPLE_RATE)

# Inspect ConvUMM Direct-Stage3

In [ ]:
MODEL_KEY = "convumm_direct_stage3_2255_2375"
ckpt_path = MODELS_DICT[MODEL_KEY]
unique_cache_dir = "./." + MODEL_KEY

In [ ]:
# Load model
token_model_conv1D = load_model_convumm_conv1D(ckpt_path, unique_cache_dir)

In [ ]:
dummy_batch = {'audio': audio_list[0].to(device), 'token': token_list[0].to(device)}
dummy_input_dict = token_model_conv1D.prepare_feature(dummy_batch)
output_dict = token_model_conv1D.model(dummy_input_dict)
print(dummy_input_dict['mel'].shape)

In [ ]:
for i in range(NUM_PLOTS):
    plt.figure(figsize=(16, 8))
    # plt.pcolor(dummy_input_dict['mel'][i].detach().T.cpu().numpy(), vmin=-6, vmax=0.5)
    plt.pcolor(dummy_input_dict['mel'][i].detach().T.cpu().numpy())
    plt.title("Ground Truth Mel-160 Spec Input")

In [ ]:
print(dummy_input_dict.keys())
print(output_dict.keys())
print(output_dict['mel_out'].shape)

In [ ]:
for i in range(NUM_PLOTS):
    plt.figure(figsize=(16, 8))
    plt.pcolor(output_dict['mel_out'][i].detach().T.cpu().numpy())
    plt.title(MODEL_KEY + " Mel-160 Spec Reconstruction")

# Load Existing WAV audio

In [ ]:
# 2JUN2024 I use audio files from this Lark Doc:
# https://bytedance.sg.larkoffice.com/docx/KJx5dHWBooPcV9x46GZleN3dgdd
audio_list = []
for i in range(5):
    audio_file = f"recipes/umm/notebooks/download_{i}.wav"
    waveform, sample_rate = torchaudio.load(audio_file)
    audio_list.append(waveform)
audio_input_loaded = torch.cat(audio_list).unsqueeze(1)
token_list_loaded = token_list[0][0:5]
print(audio_input_loaded.shape)
print(token_list_loaded.shape)

In [ ]:
ipd.Audio(audio_input_loaded[0], rate=SAMPLE_RATE)

In [ ]:
ipd.Audio(audio_input_loaded[1], rate=SAMPLE_RATE)

In [ ]:
ipd.Audio(audio_input_loaded[2], rate=SAMPLE_RATE)

In [ ]:
ipd.Audio(audio_input_loaded[3], rate=SAMPLE_RATE)

In [ ]:
ipd.Audio(audio_input_loaded[4], rate=SAMPLE_RATE)

# Inspect ConformerUMM 

In [ ]:
MODEL_KEY = "legacy_conformerumm_svs"
ckpt_path = MODELS_DICT[MODEL_KEY]
unique_cache_dir = "./." + MODEL_KEY

In [ ]:
# Load model
token_model_conformer = load_model_conformer(ckpt_path, unique_cache_dir)

In [ ]:
dummy_batch = {'audio': audio_input_loaded.to(device), 'token': token_list_loaded.to(device)}
dummy_input_dict = token_model_conformer.prepare_feature(dummy_batch)
output_dict = token_model_conformer.model(dummy_input_dict)
print(dummy_input_dict['mel'].shape)

In [ ]:
NUM_PLOTS=5
for i in range(NUM_PLOTS):
    plt.figure(figsize=(16, 8))
    # plt.pcolor(dummy_input_dict['mel'][i].detach().T.cpu().numpy(), vmin=-6, vmax=0.5)
    plt.pcolor(dummy_input_dict['mel'][i].detach().T.cpu().numpy())
    plt.title("Ground Truth Mel-128 Spec Input")

In [ ]:
print(dummy_input_dict.keys())
print(output_dict.keys())
print(output_dict['mel_out'].shape)

In [ ]:
for i in range(NUM_PLOTS):
    plt.figure(figsize=(16, 8))
    plt.pcolor(output_dict['mel_out'][i].detach().T.cpu().numpy())
    plt.title(MODEL_KEY + " Mel-128 Spec Reconstruction")

# Inspect ConvUMM-Pitch

In [ ]:
MODEL_KEY = "convumm_spl" # "convumm_spl_ppl", "convumm_spl", "convumm_ppl"
ckpt_path = MODELS_DICT[MODEL_KEY]
unique_cache_dir = "./." + MODEL_KEY

In [ ]:
# Load model
token_model_convumm_pitch = load_model_convumm_pitch(ckpt_path, unique_cache_dir)

In [ ]:
dummy_batch = {'audio': audio_input_loaded.to(device), 'token': token_list_loaded.to(device)}
dummy_input_dict = token_model_convumm_pitch.prepare_feature(dummy_batch)
output_dict = token_model_convumm_pitch.model(dummy_input_dict)
print(dummy_input_dict['mel'].shape)

In [ ]:
for i in range(NUM_PLOTS):
    plt.figure(figsize=(16, 8))
    # plt.pcolor(dummy_input_dict['mel'][i].detach().T.cpu().numpy(), vmin=-6, vmax=0.5)
    plt.pcolor(dummy_input_dict['mel'][i].detach().T.cpu().numpy())
    plt.title("Ground Truth Mel-160 Spec Input")

In [ ]:
print(dummy_input_dict.keys())
print(output_dict.keys())
print(output_dict['mel_out'].shape)

In [ ]:
for i in range(NUM_PLOTS):
    plt.figure(figsize=(16, 8))
    plt.pcolor(output_dict['mel_out'][i].detach().T.cpu().numpy())
    plt.title(MODEL_KEY + " Mel-160 Spec Reconstruction")